# Modul 18: CNNs und Transfer Learning mit PyTorch | Übungen

## Überblick

Sie erstellen Bild-Datasets und DataLoader, implementieren ein LeNet-ähnliches CNN und analysieren Fehlerbilder. Danach testen Sie Augmentation, Batch Normalization, Dropout, Lernratensteuerung und Early Stopping sowie MobileNetV3 Small als eingefrorenen Transfer-Learning-Extraktor.

**Zugehörige Vorlesungen**

- **LeNet mit PyTorch**
- **Transfer mit PyTorch**

## Lernziele

Nach der Bearbeitung können Sie:

- kleine Bilddaten als kanalzuerst angeordnete PyTorch-Tensoren mit getrennten Trainings- und Evaluationstransformationen vorbereiten.
- ein LeNet-ähnliches nn.Module trainieren und mit Konfusionsmatrix sowie Fehlbildern bewerten.
- Regularisierung und Transfer Learning kontrolliert einsetzen und Modelle nach Leistung, Laufzeit und Größe vergleichen.

## Geprüfte Fähigkeiten

- Bild-Dataset, DataLoader, Augmentation und Conv2d-Formen
- PyTorch-CNN, BatchNorm, Dropout, Scheduler, Early Stopping und Fehleranalyse
- torchvision MobileNetV3 Small, Einfrieren, Kopftraining, Inferenzzeit und Offline-Fallback

## Hinweise zur Bearbeitung

Dieses Notebook dient als praktische Übung und Lernstandskontrolle. Führen Sie zuerst die Einrichtungszelle aus und bearbeiten Sie danach die Aufgaben in der angegebenen Reihenfolge. Die vorgesehenen Arbeitsbereiche sind deutlich markiert.

- **Erwarteter Schwierigkeitsgrad:** anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt den kleinen Digits-Datensatz, teilt ihn reproduzierbar und wandelt die Pixel in Float32-Tensoren im Bereich 0 bis 1 um. Alle Standardaufgaben laufen auf der CPU. Der optionale Download vortrainierter MobileNet-Gewichte besitzt einen dokumentierten Offline-Fallback.

In [ ]:
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision import models

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
geraet = torch.device("cpu")

ziffern = load_digits()
bilder_gesamt = (ziffern.images.astype("float32") / 16.0)
labels_gesamt = ziffern.target.astype("int64")

bilder_train_np, bilder_test_np, labels_train_np, labels_test_np = train_test_split(
    bilder_gesamt,
    labels_gesamt,
    test_size=0.20,
    stratify=labels_gesamt,
    random_state=RANDOM_SEED,
)
bilder_train_np, bilder_val_np, labels_train_np, labels_val_np = train_test_split(
    bilder_train_np,
    labels_train_np,
    test_size=0.20,
    stratify=labels_train_np,
    random_state=RANDOM_SEED,
)

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Train, Validierung, Test:", bilder_train_np.shape, bilder_val_np.shape, bilder_test_np.shape)

### Aufgabe 1: Trainings- und Evaluationstransformationen trennen

Definieren Sie eine Dataset-Klasse für die 8-mal-8-Bilder. Jedes Bild soll als Tensor der Form `(1, 8, 8)` zurückgegeben werden. Im Trainingsmodus sollen mit jeweils kleiner Wahrscheinlichkeit eine Verschiebung um höchstens ein Pixel und schwaches Gaußrauschen angewendet werden. Werte müssen anschließend auf 0 bis 1 begrenzt werden.

Validierung und Test dürfen keine zufällige Augmentation erhalten. Erstellen Sie Loader mit Batchgröße 32 und visualisieren Sie ein Originalbild sowie drei augmentierte Varianten desselben Trainingsbildes.

In [ ]:
class DigitsDataset(Dataset):
    def __init__(self, bilder, labels, augment=False, seed=42):
        pass

    def __len__(self):
        pass

    def __getitem__(self, index):
        pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum darf eine zufällige Augmentation nicht auf Validierungs- und Testbilder angewendet werden?

### Aufgabe 2: Conv2d-Formen verfolgen und LeNet definieren

Wenden Sie eine `nn.Conv2d(1, 16, kernel_size=3, padding=1)` und danach `nn.MaxPool2d(2)` auf einen Batch an. Geben Sie die Formen aus.

Definieren Sie anschließend eine Klasse `KleinesLeNet` mit zwei Conv-Blöcken, ReLU, Pooling, Flatten, einer Dense-Schicht und zehn Logits. Das Modell soll optional BatchNorm und Dropout verwenden. Prüfen Sie die Ausgabeform und ermitteln Sie die Zahl trainierbarer Parameter.

In [ ]:
class KleinesLeNet(nn.Module):
    def __init__(self, use_regularization=False):
        super().__init__()
        pass

    def forward(self, x):
        pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum ist es sinnvoll, die Flatten-Größe vor der Dense-Schicht ausdrücklich herzuleiten?

### Aufgabe 3: LeNet auf der CPU trainieren und Fehler analysieren

Schreiben Sie kompakte Trainings- und Auswertungsfunktionen für die Mehrklassenklassifikation mit `CrossEntropyLoss`. Trainieren Sie das unregularisierte LeNet mit Adam für höchstens zwölf Epochen und speichern Sie den besten Validierungszustand.

Berichten Sie Test-Accuracy, Balanced Accuracy und Macro-F1. Erstellen Sie eine Konfusionsmatrix und visualisieren Sie bis zu sechs falsch klassifizierte Testbilder mit ihren Softmax-Sicherheiten.

In [ ]:
def trainiere_cnn_epoche(modell, loader, optimizer, loss_fn, device):
    pass

def bewerte_cnn(modell, loader, loss_fn, device):
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum sollte für CrossEntropyLoss keine Softmax-Schicht im Modell stehen?

### Aufgabe 4: Regularisierung, Scheduler und Early Stopping kontrolliert einsetzen

Trainieren Sie die regularisierte LeNet-Variante mit BatchNorm und Dropout. Nutzen Sie Adam, `ReduceLROnPlateau` auf dem Validierungsverlust und Early Stopping mit Geduld 4. Speichern Sie pro Epoche die Lernrate.

Vergleichen Sie unregularisiertes und regularisiertes Modell hinsichtlich bester Validierungsverlust, Test-Macro-F1 und Anzahl trainierbarer Parameter. Erklären Sie, weshalb BatchNorm im Evaluationsmodus wichtig ist.

In [ ]:
# Verwenden Sie KleinesLeNet(use_regularization=True).

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum muss BatchNorm beim Bewerten in model.eval() geschaltet werden?

### Aufgabe 5: MobileNetV3 Small einfrieren und einen Kopf trainieren

Skalieren Sie kleine Teilmengen der Digits-Bilder auf 64-mal-64, wiederholen Sie den Graustufenkanal dreimal und normalisieren Sie mit den ImageNet-Mittelwerten und -Standardabweichungen.

Versuchen Sie `MobileNet_V3_Small_Weights.DEFAULT` zu laden. Bei fehlendem Internet muss der Code auf zufällige Gewichte zurückfallen und dies dokumentieren. Frieren Sie alle Basisparameter ein, ersetzen Sie die letzte Klassifikationsschicht durch zehn Ausgaben und trainieren Sie nur den neuen Kopf für höchstens zwei Epochen.

In [ ]:
VERSUCHE_VORTRAINIERTE_GEWICHTE = True

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welche Einschränkung besitzt der eingefrorene Offline-Fallback?

### Aufgabe 6: Integrationsaufgabe: Genauigkeit, Laufzeit und Modellgröße vergleichen

Trainieren Sie eine Pixel-LogReg-Baseline auf den flach dargestellten Digits. Vergleichen Sie Pixel-LogReg, das beste LeNet und MobileNetV3 anhand von Accuracy, Balanced Accuracy, Macro-F1, Gesamtparametern, trainierbaren Parametern und mittlerer Inferenzzeit pro Beispiel.

Messen Sie die Inferenzzeit nach einem Aufwärmdurchlauf mindestens fünfmal. Dokumentieren Sie bei MobileNetV3, ob echte vortrainierte Gewichte verwendet wurden. Treffen Sie eine begründete Wahl für einen CPU-basierten Offline-Einsatz.

In [ ]:
def messe_inferenzzeit(modell, beispiel_batch, wiederholungen=5):
    pass

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welcher Ansatz ist für einen kleinen Offline-CPU-Einsatz am überzeugendsten?

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?